In [12]:
import json
import pandas as pd
from google.cloud import firestore

PROCESSING_FILE_PATH = 'resources/acts_2024_v2.json'

try:
    # Replace 'path/to/credentials.json' with the path to your credentials file
    db = firestore.Client.from_service_account_json('resources/legal-research-platform-firebase-adminsdk-fbsvc-46a9ab3605.json', project='legal-research-platform')
    print("Successfully connected to Firestore!")
    # You can optionally print your project ID to confirm the connection
    print(f"Project ID: {db.project}")
except Exception as e:
    print(f"Error connecting to Firestore: {e}")


if 'db' not in globals():
    print("Firestore client 'db' is not initialized. Please run the Firestore connection cell first.")
else:
    # Reference to the 'summaries' collection
    summaries_ref = db.collection('summaries')

    # Load the cases data
    try:
        with open(PROCESSING_FILE_PATH, 'r', encoding='utf-8') as f:
            acts_data = json.load(f)
        print(f"Loaded {len(acts_data)} acts from {PROCESSING_FILE_PATH}")

        # Process and upload each case
        for act in acts_data:
            structured_sections = act.get('structuredSections', [])
            for section in structured_sections:
                section_no = section.get('section')
                section_text = section.get('text')

                if not section_no or not section_text:
                    continue

                # Create an object to store in the document database
                summary_data = {
                    'doc_id': act['id'],
                    'doc_type': 'act',
                    'section_no': section_no,
                    'summary_text': section_text,
                    'word_count': len(section_text.split()),
                    'timestamp': firestore.SERVER_TIMESTAMP,
                }

                # Overwrite an existing document with the same ID
                doc_ref = summaries_ref.document(f"{act['id']}_section{section_no}")
                doc_ref.set(summary_data, merge=True)
                print(f"Summary for section {section_no} stored successfully with document ID: {doc_ref.id}")

    except FileNotFoundError:
        print(f"Error: File not found at {PROCESSING_FILE_PATH}")
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {PROCESSING_FILE_PATH}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

Successfully connected to Firestore!
Project ID: legal-research-platform
Loaded 32 acts from resources/acts_2024_v2.json
Summary for section 1 stored successfully with document ID: a777ce4f-de7a-48f2-9bde-64bca381d76f_section1
Summary for section 2 stored successfully with document ID: a777ce4f-de7a-48f2-9bde-64bca381d76f_section2
Summary for section 1 stored successfully with document ID: 9647a713-7fc1-489b-a8ae-7c6375d8bffc_section1
Summary for section 2 stored successfully with document ID: 9647a713-7fc1-489b-a8ae-7c6375d8bffc_section2
Summary for section 3 stored successfully with document ID: 9647a713-7fc1-489b-a8ae-7c6375d8bffc_section3
Summary for section 4 stored successfully with document ID: 9647a713-7fc1-489b-a8ae-7c6375d8bffc_section4
Summary for section 1 stored successfully with document ID: db16d159-9864-4fb5-a7e8-8a5dbaa29fd4_section1
Summary for section 2 stored successfully with document ID: db16d159-9864-4fb5-a7e8-8a5dbaa29fd4_section2
Summary for section 3 stored su